# 🧪 BoxBunny Unit Tests

> **Version:** 1.0.0  
> **Last Updated:** February 2026  
> **Package:** boxing_robot_ws

Interactive unit tests for validating BoxBunny system components. Each test launches a specific subsystem for manual verification.

---

## 📋 Test Index

| Test | Component | Description |
|------|-----------|-------------|
| 1 | GUI Design | Verify main interface layout and navigation |
| 2 | IMU Calibration | Test accelerometer/gyroscope data and calibration |
| 3 | Reaction Drill | Full drill flow with pose detection |
| 4 | Punch Detection | Color tracking punch detection |
| 5 | Height Calibration | Pose + depth height measurement |
| 6 | LLM Chat | Interactive AI coach conversation |

---

## ⚠️ Prerequisites

- **Conda environment:** `boxing_ai` must be active
- **ROS 2 Humble:** Installed and sourced
- **Hardware:** RealSense camera connected (for vision tests)
- **GPU (optional):** Required for action model tests

---

## 🔧 Setup Environment

**Run this cell first** to configure the shell environment for all subsequent tests.

In [ ]:
%%bash
# =============================================================================
# BoxBunny Test Environment Setup
# Creates a reusable shell script for all test cells
# =============================================================================

cat > /tmp/boxbunny_env.sh <<'SH'
# Conda environment activation
source ~/miniconda3/etc/profile.d/conda.sh
conda activate boxing_ai

# Prevent site-packages conflicts
export PYTHONNOUSERSITE=1

# Library paths
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH
export SETUPTOOLS_USE_DISTUTILS=stdlib

# ROS 2 setup
source /opt/ros/humble/setup.bash
source /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/install/setup.bash 2>/dev/null || true
SH

echo "✅ Test environment ready"

## 🔨 Build Workspace

**Required:** Compiles `boxbunny_msgs` and all other packages before running tests.

In [ ]:
%%bash
# =============================================================================
# Build BoxBunny Workspace
# =============================================================================

source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws

echo "🔨 Building workspace..."
colcon build --symlink-install
source install/setup.bash

echo "✅ Build complete"

---

## Test 1️⃣: GUI Design

**Purpose:** Verify the main BoxBunny GUI appearance and layout.

### Expected Behavior
- ✅ Main window opens in fullscreen or maximized
- ✅ Navigation between pages works (Home, Drills, Settings)
- ✅ All buttons and labels render correctly
- ✅ Dark theme applied consistently

### Exit
Close the window or press `Ctrl+C` in the terminal.

In [ ]:
%%bash
# =============================================================================
# Test 1: Main GUI
# Launches the BoxBunny GUI directly (uses conda Python for PySide6)
# =============================================================================

source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/src/boxbunny_gui/boxbunny_gui

echo "🖥️  Launching Main GUI..."
python gui_main.py

---

## Test 2️⃣: IMU Calibration

**Purpose:** Test the IMU sensors and punch calibration interface.

### Hardware Requirements
- MPU6050 IMU connected via I2C (address 0x68)

### Expected Behavior
- ✅ IMU Calibration GUI opens
- ✅ Real-time accelerometer/gyroscope values displayed
- ✅ 3D axis visualization updates smoothly
- ✅ "Calibrate Strike Impact" button triggers calibration flow
- ✅ Punch detection works after calibration

### Exit
Close the window or press `Ctrl+C` in the terminal.

In [ ]:
%%bash
# =============================================================================
# Test 2: IMU Calibration GUI
# Starts IMU nodes and launches calibration interface
# =============================================================================

source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws

# Rebuild IMU package for any config changes
echo "🔨 Building boxbunny_imu..."
colcon build --packages-select boxbunny_imu --symlink-install > /dev/null 2>&1
source install/setup.bash

# Start IMU nodes in background
echo "📡 Starting IMU nodes..."
ros2 run boxbunny_imu mpu6050_node &
ros2 run boxbunny_imu imu_punch_classifier &
sleep 1

# Run calibration GUI
echo "🖥️  Launching IMU Calibration GUI..."
cd src/boxbunny_imu/boxbunny_imu
python imu_punch_gui.py

---

## Test 3️⃣: Reaction Drill

**Purpose:** Test the full reaction time drill flow using Pose/Action model.

### Hardware Requirements
- RealSense RGB-D camera connected
- GPU recommended for action model

### Expected Behavior
1. ✅ System starts automatically (Drill Manager, AI, GUI)
2. ✅ Video feed with skeleton overlay appears
3. ✅ Click **START REACTION DRILL** to begin (3 trials)
4. ✅ Flow per trial: `Wait...` → `PUNCH!` → (you punch) → `✓` → repeat
5. ✅ Punching before "PUNCH!" shows **"EARLY!"** penalty
6. ✅ After 3 trials, shows average reaction time

### Exit
Close the window or press `Ctrl+C` in the terminal.

In [ ]:
%%bash
# =============================================================================
# Test 3: Reaction Drill
# Launches the unified reaction drill with pose detection
# =============================================================================

source /tmp/boxbunny_env.sh

# Kill any conflicting camera nodes first
echo "🧹 Cleaning up existing processes..."
pkill -f realsense2_camera_node 2>/dev/null || true
pkill -f live_infer_rgbd.py 2>/dev/null || true
pkill -f "ros2 launch boxbunny_bringup" 2>/dev/null || true
sleep 2

cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws

# Rebuild drills package
echo "🔨 Building boxbunny_drills..."
colcon build --packages-select boxbunny_drills --symlink-install > /dev/null
source install/setup.bash

# Launch unified reaction drill
echo "🎯 Launching Reaction Drill..."
ros2 launch boxbunny_drills reaction_drill.launch.py

---

## Test 4️⃣: Punch Detection (Color Tracking)

**Purpose:** Test standalone color-based punch detection.

### Glove Colors
| Glove | Color | Punch Type |
|-------|-------|------------|
| LEFT | 🔴 Red | JAB |
| RIGHT | 🟢 Green | CROSS |

### Hardware Requirements
- RealSense RGB-D camera connected
- Colored gloves or wristbands

### Expected Behavior
1. ✅ Punch Detection GUI opens with live camera feed
2. ✅ Bounding boxes appear around detected gloves
3. ✅ Punch detected when glove approaches camera quickly
4. ✅ Punch counter and log show all detections

### Exit
Close the window or press `Ctrl+C` in the terminal.

In [ ]:
%%bash
# =============================================================================
# Test 4: Punch Detection (Color Tracking)
# Standalone color-based punch detection test
# =============================================================================

source /tmp/boxbunny_env.sh

# Kill any existing camera processes
echo "🧹 Cleaning up existing processes..."
pkill -f realsense2_camera_node 2>/dev/null || true
pkill -f live_infer_rgbd.py 2>/dev/null || true
pkill -f punch_detection_gui.py 2>/dev/null || true
sleep 1

cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws

# Rebuild drills package
echo "🔨 Building boxbunny_drills..."
colcon build --packages-select boxbunny_drills --symlink-install > /dev/null
source install/setup.bash

# Launch punch detection
echo "🎨 Launching Punch Detection (Color Tracking)..."
ros2 launch boxbunny_drills punch_detection.launch.py

---

## Test 5️⃣: Height Calibration

**Purpose:** Calibrate and verify user height using Pose + Depth data.

> **Note:** This test is completely standalone - runs its own camera and pose model.

### Hardware Requirements
- RealSense RGB-D camera connected
- GPU recommended for pose estimation

### Setup Instructions
1. Stand 2-3 meters from camera
2. Ensure full body is visible (head to feet)
3. Stand straight in neutral pose

### Expected Behavior
1. ✅ Height Calibration GUI opens with live camera feed
2. ✅ Skeleton overlay and bounding box appear on your body
3. ✅ Click **CALIBRATE HEIGHT** and hold still (2 second countdown)
4. ✅ System displays calibrated height in meters

### Exit
Close the window or press `Ctrl+C` in the terminal.

In [ ]:
%%bash
# =============================================================================
# Test 5: Height Calibration
# Standalone height measurement using pose + depth
# =============================================================================

source /tmp/boxbunny_env.sh

# Kill any existing camera processes
echo "🧹 Cleaning up existing processes..."
pkill -f realsense2_camera_node 2>/dev/null || true
pkill -f live_infer_rgbd.py 2>/dev/null || true
pkill -f height_calibration_gui.py 2>/dev/null || true
sleep 1

cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws

# Rebuild drills package
echo "🔨 Building boxbunny_drills..."
colcon build --packages-select boxbunny_drills --symlink-install > /dev/null
source install/setup.bash

# Launch height calibration
echo "📏 Launching Height Calibration..."
ros2 launch boxbunny_drills height_calibration.launch.py

---

## Test 6️⃣: LLM Chat GUI

**Purpose:** Test the interactive AI coach conversation interface.

### Requirements
- LLM model file available (Qwen2.5 or similar GGUF)
- Sufficient RAM (8GB+ recommended)

### Expected Behavior
1. ✅ LLM Chat GUI opens
2. ✅ Model loads (may take 10-30 seconds)
3. ✅ Type messages and receive AI responses
4. ✅ Streaming output shows tokens as they generate

### Exit
Close the window.

In [ ]:
%%bash
# =============================================================================
# Test 6: LLM Chat GUI
# Interactive AI coach conversation interface
# =============================================================================

source /tmp/boxbunny_env.sh
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/src/boxbunny_llm/boxbunny_llm

echo "🤖 Launching LLM Chat GUI..."
echo "   (Model loading may take 10-30 seconds)"
python llm_chat_gui.py

---

## 📋 Test Summary

After running all tests, verify each component passed:

| Component | Status | Notes |
|-----------|--------|-------|
| Main GUI | ⬜ | Navigation, styling |
| IMU Calibration | ⬜ | Sensor data, calibration flow |
| Reaction Drill | ⬜ | Full drill loop, timing |
| Punch Detection | ⬜ | Color tracking, velocity |
| Height Calibration | ⬜ | Pose + depth accuracy |
| LLM Chat | ⬜ | Model loading, responses |

---

*BoxBunny Training System - Unit Tests v1.0.0*